# Credit Scoring Orquestador

Notebook orquestador del homework. Toda la logica de negocio vive en `src/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import json

import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import load_raw, validate_schema, create_features
from src.features import select_features_by_iv, build_woe_tables, transform_woe
from src.models import train_all_models, evaluate_models, save_model
from src.metrics import auc_roc, costo_total, build_scorecard

## 1. Carga y validacion

In [3]:
data_path = PROJECT_ROOT / "data" / "raw" / "bankloan.csv"
assert data_path.exists(), "No existe data/raw/bankloan.csv en el repositorio"
assert data_path.suffix == ".csv", "El dataset de entrada debe estar en formato CSV"

df = load_raw(str(data_path))
assert not df.empty, "El dataset cargado esta vacio"
validate_schema(df)
df = create_features(df)
assert not df.empty, "El dataset quedo vacio despues del preprocesamiento"
df.head()

,customer,Age,Education,Employ,Address,Income,Leverage,Creddebt,OthDebt,MonthlyLoad,Default,OthDebtRatio
0,10012,28,Med,7,2.0,44.0,17.7,2.99,4.80,0.58,0,0.109091
1,10017,64,Posg,34,17.0,116.0,14.7,5.05,12.00,0.27,0,0.103448
2,10030,40,Bas,20,12.0,61.0,4.8,1.04,1.89,0.13,0,0.030984
3,10039,30,Bas,11,3.0,27.0,34.5,1.75,7.56,1.62,0,0.280000
4,10069,25,Bas,2,2.0,30.0,22.4,0.76,5.96,0.97,1,0.198667


In [4]:
feature_df = df.drop(columns=["customer"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    feature_df.drop(columns=["Default"]),
    feature_df["Default"],
    test_size=0.2,
    random_state=42,
    stratify=feature_df["Default"],
)

print(X_train.shape, X_test.shape)

(1172, 10) (294, 10)


## 2. Seleccion de variables por IV

In [5]:
train_with_target = X_train.assign(Default=y_train)
features_sel = select_features_by_iv(
    train_with_target,
    target="Default",
    threshold=0.1,
)
assert features_sel, "No se seleccionaron variables con el umbral de IV actual"
features_sel

['Age',
 'Employ',
 'Address',
 'Income',
 'Leverage',
 'Creddebt',
 'MonthlyLoad',
 'OthDebtRatio']

## 3. Transformacion WoE

In [6]:
woe_tables = build_woe_tables(
    train_with_target,
    features_sel,
    target="Default",
)

X_train_woe = transform_woe(X_train[features_sel], woe_tables)
X_test_woe = transform_woe(X_test[features_sel], woe_tables)

X_train_woe.head()

,Age_woe,Employ_woe,Address_woe,Income_woe,Leverage_woe,Creddebt_woe,MonthlyLoad_woe,OthDebtRatio_woe
368,0.997858,0.527855,0.668379,0.225643,0.385857,-0.406136,0.698175,0.596146
748,0.997858,0.148365,0.668379,0.106885,-0.195643,0.056875,0.108763,0.210822
86,0.107543,0.148365,0.303206,0.058753,-1.464743,0.056875,-1.293174,-0.937580
337,0.107543,0.108168,-0.041942,0.058753,0.385857,0.132861,0.698175,0.056875
806,0.107543,0.527855,-0.041942,0.225643,-0.471733,-0.972819,-0.395983,0.056875


In [7]:
scorecard_df = build_scorecard(woe_tables)
scorecard_df.head()

,feature,bin,total,n_events,n_non_events,woe,iv_bin,points,riesgo_relativo
0,Age,"(17.999, 19.1]",118,71,47,-0.972819,0.100846,19.46,alto
1,Age,"(19.1, 22.0]",130,65,65,-0.560286,0.036671,11.21,alto
2,Age,"(28.0, 30.0]",82,39,43,-0.462648,0.015688,9.25,alto
3,Age,"(22.0, 25.0]",135,61,74,-0.367095,0.016151,7.34,alto
4,Age,"(25.0, 28.0]",127,55,72,-0.290953,0.009483,5.82,alto


## 4. Entrenamiento y evaluacion

In [8]:
models = train_all_models(X_train_woe, y_train)
tabla_auc = evaluate_models(models, X_test_woe, y_test)
tabla_auc

c:\Users\litoa\OneDrive\Documents\GitHub\ml_finance_carenas\.conda\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\litoa\OneDrive\Documents\GitHub\ml_finance_carenas\.conda\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\litoa\OneDrive\Documents\GitHub\ml_finance_carenas\.conda\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead

,Modelo,AUC_ROC
0,Logistic Regression,0.824054
1,XGBoost,0.822605
2,Random Forest,0.819006


In [9]:
mejor_nombre = tabla_auc.iloc[0]["Modelo"]
mejor_modelo = models[mejor_nombre]

metadata = {
    "model_name": mejor_nombre,
    "version": "1.0",
    "author": "Arenas",
    "dataset": "data/raw/bankloan.csv",
    "n_train_samples": int(len(y_train)),
    "n_test_samples": int(len(y_test)),
    "features_selected": list(X_train_woe.columns),
    "hyperparameters": mejor_modelo.get_params(),
    "metrics": {
        "auc_test": round(auc_roc(mejor_modelo, X_test_woe, y_test), 4),
        "costo_total_test": int(
            costo_total(
                y_test.to_numpy(),
                mejor_modelo.predict_proba(X_test_woe)[:, 1],
                umbral=0.5,
            )
        ),
    },
}

model_path = save_model(
    mejor_modelo,
    path=str(PROJECT_ROOT / "models" / "baseline_v1"),
    metadata=metadata,
)
assert model_path.exists(), "El archivo del modelo no fue generado"
model_path

WindowsPath('C:/Users/litoa/OneDrive/Documents/GitHub/ml_finance_carenas/credit_scoring_arenas/models/baseline_v1/logistic_regression.pkl')

## 5. Verificacion de metadata

In [10]:
metadata_path = PROJECT_ROOT / "models" / "baseline_v1" / "metadata.json"

assert metadata_path.exists(), "metadata.json no existe, revisa save_model()"

with open(metadata_path, encoding="utf-8") as f:
    meta = json.load(f)

campos_requeridos = [
    "model_name",
    "version",
    "saved_at",
    "author",
    "dataset",
    "n_train_samples",
    "n_test_samples",
    "features_selected",
    "hyperparameters",
    "metrics",
]

for campo in campos_requeridos:
    assert campo in meta, f"Campo faltante en metadata.json: {campo}"

metricas_requeridas = ["auc_test", "costo_total_test"]
for metrica in metricas_requeridas:
    assert metrica in meta["metrics"], f"Metrica faltante en metadata.json: {metrica}"

assert meta["author"] == "Arenas", "El author de metadata.json no coincide"
assert meta["dataset"] == "data/raw/bankloan.csv", "La ruta del dataset en metadata.json no coincide"
assert meta["n_train_samples"] + meta["n_test_samples"] == len(df), "El total de muestras en metadata.json no coincide con el dataset procesado"
assert isinstance(meta["features_selected"], list) and meta["features_selected"], "features_selected debe ser una lista no vacia"
assert isinstance(meta["hyperparameters"], dict) and meta["hyperparameters"], "hyperparameters debe ser un diccionario no vacio"

pd.DataFrame([meta])

,model_name,version,author,dataset,n_train_samples,n_test_samples,features_selected,hyperparameters,metrics,saved_at
0,Logistic Regression,1.0,Arenas,data/raw/bankloan.csv,1172,294,"[Age_woe, Employ_woe, Address_woe, Income_woe,...","{'C': 1.0, 'class_weight': None, 'dual': False...","{'auc_test': 0.8241, 'costo_total_test': 20800}",2026-05-04
